# Find efterspørgselsdrivere med PROC GLMSELECT: Trinvis udvælgelse, LASSO og valideret fremadrettet udvælgelse

## Resumé

Et retailanalyseteam bruger **PROC GLMSELECT** til at finde ud af, hvilke markedsførings- og prishåndtag der reelt flytter det ugentlige enhedssalg, og adskiller de sande efterspørgselsdrivere fra støj. Trinvis udvælgelse scoret med SBC, LASSO med 5-fold krydsvalidering og en reservedata-valideret fremadrettet søgning genfinder alle **det samme sæt af sande drivere** — egen pris, konkurrentpris, reklameudgift, e-mailvolumen, kampagner, helligdage, Northeast-regionen og Online-kanalen — og hver af de fire plantede lokkevariable (`temp_f`, `noise1`, `noise2`, `noise3`) frasorteres automatisk.

Metoderne er også tæt enige om størrelsesordenerne: hver af dem estimerer en egen-pris-effekt tæt på **-28 enheder pr. dollar** og en konkurrentpris-effekt tæt på **+14**, de substitutionsfortegn som den datagenererende ligning var bygget med. Den eneste uenighed ligger i marginen — den krydsvaliderede LASSO beholder desuden en lille, statistisk ubetydelig `Region=Midwest`-kontrast (estimat -8,3 med en standardfejl på 8,3), som både trinvis og fremadrettet udvælgelse dropper. En driverliste, der overlever tre forskellige udvælgelsesfilosofier, er langt mere troværdig end en, der er tilpasset én enkelt regel.

## Datakilder

Alle data i denne notebook er **syntetiske** og genereret direkte i koden (ingen eksterne filer, seed `20260531`). De efterligner én sæson af butiks-uge-efterspørgselspaneler for en forbrugsvareforhandler.

| Datasæt | Rækker | Granularitet | Nøglevariable |
|---------|------|-------|---------------|
| `demand` | 100 | Butik-uge | `units` (respons: ugentligt enhedssalg); `price`, `competitor` (egen & konkurrerende hyldepris); `adspend`, `email` (mediepres); `promo`, `holiday` (hændelsesflag); `region`, `channel` (CLASS-effekter); `temp_f`, `noise1`-`noise3` (lokke-/irrelevante prædiktorer) |

`units` er bygget ud fra en kendt lineær ligning, så det "korrekte" sæt af drivere kan verificeres; `temp_f` og de tre `noise`-kolonner bærer intet reelt signal og findes for at teste, om hver udvælgelsesmetode frasorterer dem.

# Find efterspørgselsdrivere med PROC GLMSELECT

En kategorichef har dusinvis af kandidatforklaringer til det ugentlige salg: egen pris, konkurrentpris, reklameudgift, e-mailvolumen, kampagner, helligdage, butiksregion, salgskanal, endda vejret. At smide dem alle ind i én regression indbyder til overfitting og ustabile koefficienter. **PROC GLMSELECT** automatiserer søgningen efter en sparsom model og understøtter trinvis udvælgelse, LASSO, elastic net og least-angle-udvælgelse med indbygget krydsvalidering og opdeling i reservedata.

I denne notebook vil vi:

1. Generere et realistisk syntetisk butiks-uge-efterspørgselspanel med et *kendt* sæt af sande drivere (samt bevidste lokkevariable).
2. Køre **trinvis udvælgelse** scoret med SBC.
3. Køre **LASSO** med 5-fold krydsvalidering.
4. Køre **fremadrettet udvælgelse valideret på 30% reservedata**.

En god udvælgelsesmetode bør genfinde de reelle drivere og frasortere støjen — lad os se.

## 1. Generer et syntetisk efterspørgselspanel

Responsvariablen `units` konstrueres ud fra en eksplicit lineær ligning, så vi kender facit: pris og konkurrentpris, reklameudgift, e-mailvolumen, kampagne- og helligdagsflagene samt region- og kanaleffekterne betyder alle noget. Variablene `temp_f`, `noise1`, `noise2` og `noise3` er rene lokkevariable uden reel sammenhæng med salget. Et `call streaminit`-seed gør dataene reproducerbare.

In [1]:
/* ---------------------------------------------------------------
   Syntetisk butiks-uge efterspørgselspanel for en forbrugsvareforhandler.
   'units' følger en KENDT ligning; temp_f og noise1-3 er lokkevariable.
   --------------------------------------------------------------- */
data demand;
    CALL streaminit(20260531);
    LÆNGDE region $9 channel $8 promo $3;
    GØR store_week = 1 TIL 100;
        /* Regionsfordeling */
        u1 = rand('uniform');
        HVIS u1 < 0.34 SÅ region = 'Northeast';
        ELLERS HVIS u1 < 0.67 SÅ region = 'Midwest';
        ELLERS region = 'South';

        /* Salgskanal */
        HVIS rand('uniform') < 0.45 SÅ channel = 'Store';
        ELLERS channel = 'Online';

        /* Pris- og mediedrivere */
        price      = round(8  + rand('uniform') * 12, 0.01);
        competitor = round(8  + rand('uniform') * 12, 0.01);
        adspend    = round(rand('gamma', 2) * 500, 1);
        email      = round(rand('uniform') * 100, 1);

        /* Hændelsesflag og en irrelevant vejr-lokkevariabel */
        temp_f     = round(40 + rand('uniform') * 50, 0.1);
        holiday    = (rand('uniform') < 0.12);
        HVIS rand('uniform') < 0.40 SÅ promo = 'Yes';
        ELLERS promo = 'No';

        /* Rene støjvariable (ingen reel effekt) */
        noise1 = rand('normal');
        noise2 = rand('normal');
        noise3 = rand('normal');

        /* Ugentligt solgte enheder fra en kendt strukturel ligning */
        units = 900
              - 28   * price
              + 14   * competitor
              + 0.06 * adspend
              + 1.2  * email
              + 55   * (promo = 'Yes')
              + 70   * holiday
              + 40   * (region = 'Northeast')
              + 25   * (channel = 'Online')
              + 30   * rand('normal');
        HVIS units < 0 SÅ units = 0;
        UDDATA;
    SLUT;
    BEHOLD region channel promo price competitor adspend email temp_f
         holiday noise1 noise2 noise3 units;
KØR;



NOTE: DATA demand


NOTE: Wrote demand (100 rows, 13 columns).
NOTE: DATA elapsed:
  wall  0.04 seconds
  cpu   0.04 seconds


## 2. Profiler dataene

Før modellering bekræftes det, at responsvariablen og de vigtigste kontinuerte kandidater ligger på fornuftige skalaer.

In [2]:
PROCEDURE GENNEMSNIT data=demand n mean std MIN MAX maxdec=1;
    VARIABEL units price competitor adspend email;
    MÆRKAT units="Solgte enheder (ugentligt)" price="Egen pris"
          competitor="Konkurrentpris" adspend="Reklameudgift"
          email="E-mailvolumen";
    TITEL "Ugentlig efterspørgsel og kandidatdrivere";
KØR;


                                       Ugentlig efterspørgsel og kandidatdrivere                                        

                                                  The MEANS Procedure

 Variable    Label                              N        Mean     Std Dev     Minimum     Maximum
 ------------------------------------------------------------------------------------------------
 units       Solgte enheder (ugentligt)       100       874.8       136.3       508.6      1162.3
 price       Egen pris                        100        14.0         3.4         8.0        19.9
 competitor  Konkurrentpris                   100        13.8         3.4         8.1        19.9
 adspend     Reklameudgift                    100       990.6       726.9        50.0      3358.0
 email       E-mailvolumen                    100        45.5        26.4         0.0        99.0
 ------------------------------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## 3. Trinvis udvælgelse scoret med SBC

Trinvis udvælgelse tilføjer og fjerner skiftevis effekter, her bedømt ud fra **Schwarz' bayesianske kriterium (SBC)** for både optagelsestesten (`select=sbc`) og det endelige modelvalg (`choose=sbc`). SBC straffer kompleksitet hårdere end AIC og favoriserer slankere modeller.

Centrale udsagn og valgmuligheder:

- `CLASS region channel promo / param=reference` erklærer de kategoriske prædiktorer med reference-celle-kodning.
- `selection=stepwise(select=sbc choose=sbc)` styrer søgningen.
- `details=summary` udskriver trin-for-trin-udvælgelsesoversigten; `stb` tilføjer standardiserede koefficienter, så effekter på forskellige skalaer kan sammenlignes.
- `output out=demand_scored p=predicted r=residual` gemmer prædikterede værdier og residualer til videre scoring.

Læs **Stepwise Selection Summary** som søgesporet: den starter fra den fulde model med 12 effekter og *fjerner* effekter én ad gangen, idet `noise1`, `noise2`, `temp_f`, `Region=Midwest`-kontrasten og `noise3` droppes i tur og orden, fordi hver fjernelse sænker SBC. Det, der overlever i **Parameter Estimates**-tabellen, er den valgte model.

In [3]:
PROCEDURE glmselect data=demand seed=20260531;
    KLASSE region channel promo / PARAM=reference;
    MODEL units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=stepwise(select=sbc choose=sbc)
          details=summary stb;
    MÆRKAT units="Solgte enheder (ugentligt)" region="Region" channel="Kanal"
          promo="Kampagne" price="Egen pris" competitor="Konkurrentpris"
          adspend="Reklameudgift" email="E-mailvolumen"
          temp_f="Temperatur (F)" holiday="Helligdag"
          noise1="Støj 1" noise2="Støj 2" noise3="Støj 3";
    UDDATA out=demand_scored p=predicted r=residual;
    TITEL "Trinvis udvælgelse af efterspørgselsdrivere (SBC)";
KØR;


                                       Ugentlig efterspørgsel og kandidatdrivere                                        

The GLMSELECT Procedure


Dependent Variable: UNITS Solgte enheder (ugentligt)


Number of Observations Used: 100

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                Stepwise Selection Summary                                                 

    Step    Action          Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)
--------  --------  --------------  -----------------  --------  --------  --------  --------  --------  --------  --------
       1   Removed          Støj 1                 12    0.9507    0.9439  707.0094  711.2420  713.1843  740.8766   12.0136



NOTE: PROC GLMSELECT data=demand

NOTE: The data set WORK.DEMAND_SCORED has 100 observations and 15 variables.
NOTE: PROC GLMSELECT statement used.


## 4. LASSO med krydsvalidering

LASSO trækker koefficienterne mod nul og udfører udvælgelse og regularisering samtidig. I stedet for at stoppe ved et fast kriterium lader vi **5-fold krydsvalidering** vælge det punkt på LASSO-stien med den bedste fejl uden for stikprøven.

- `selection=lasso(choose=cv stop=none)` sporer hele LASSO-stien og vælger det CV-optimale trin.
- `cvmethod=random(5)` fordeler observationerne på 5 tilfældige folds.

**LASSO Selection Summary** viser rækkefølgen, hvori effekter optages, efterhånden som straffen slækkes: `price` først, dernæst `adspend`, `competitor`, Northeast-regionen, `email`, `promo` og `holiday` — alle syv sande signaler optages, før nogen lokkevariabel gør. Krydsvalideringen vælger derefter det trin, hvor PRESS uden for stikprøven er lavest, og den resulterende **Parameter Estimates**-tabel beholder netop de reelle drivere (plus et lille `Region=Midwest`-led), mens `temp_f`, `noise1`, `noise2` og `noise3` udelukkes, da de først optages helt til sidst på stien.

In [4]:
PROCEDURE glmselect data=demand seed=20260531;
    KLASSE region channel promo / PARAM=reference;
    MODEL units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=lasso(choose=cv stop=none)
          cvmethod=RANDOM(5);
    MÆRKAT units="Solgte enheder (ugentligt)" region="Region" channel="Kanal"
          promo="Kampagne" price="Egen pris" competitor="Konkurrentpris"
          adspend="Reklameudgift" email="E-mailvolumen"
          temp_f="Temperatur (F)" holiday="Helligdag"
          noise1="Støj 1" noise2="Støj 2" noise3="Støj 3";
    TITEL "LASSO med 5-fold krydsvalidering";
KØR;


                                       Ugentlig efterspørgsel og kandidatdrivere                                        

The GLMSELECT Procedure


Dependent Variable: UNITS Solgte enheder (ugentligt)


Number of Observations Used: 100

  Cross Validation Information   

                   Item     Value
-----------------------  --------
Cross Validation Method    Random
        Number of Folds         5

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                         LASSO Selection Summary                                                          

    Step    Action            Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)        PRESS
--------  --------  ----------------  --------


NOTE: PROC GLMSELECT data=demand

NOTE: PROC GLMSELECT statement used.


## 5. Fremadrettet udvælgelse valideret på reservedata

En tredje, supplerende strategi: byg modellen ved **fremadrettet udvælgelse** (effekter kommer kun til, forlader aldrig modellen), men stop dér, hvor præstationen på en uafhængig reservedatastikprøve er bedst — det beskytter direkte mod overfitting.

- `partition fraction(validate=0.30)` reserverer tilfældigt 30% af rækkerne til validering, så der er 70 trænings- og 30 valideringsobservationer tilbage.
- `selection=forward(select=aic choose=validate)` tilføjer effekter efter AIC, men vælger den endelige model ud fra fejlen på valideringsstikprøven.

**Partition Fractions**-tabellen bekræfter 70/30-opdelingen. Fremadrettet udvælgelse tilføjer derefter effekter, indtil valideringsfejlen holder op med at forbedres; de otte effekter i den endelige **Parameter Estimates**-tabel er netop de sande drivere, mens de fire lokkevariable aldrig optages. Når tre metoder bygget på forskellige principper lander på de samme drivere, er modellen langt mere sandsynligt reel end et artefakt af én enkelt udvælgelsesregel.

In [5]:
PROCEDURE glmselect data=demand seed=20260531;
    KLASSE region channel promo / PARAM=reference;
    MODEL units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=forward(select=aic choose=validate);
    partition FRACTION(validate=0.30);
    MÆRKAT units="Solgte enheder (ugentligt)" region="Region" channel="Kanal"
          promo="Kampagne" price="Egen pris" competitor="Konkurrentpris"
          adspend="Reklameudgift" email="E-mailvolumen"
          temp_f="Temperatur (F)" holiday="Helligdag"
          noise1="Støj 1" noise2="Støj 2" noise3="Støj 3";
    TITEL "Fremadrettet udvælgelse valideret på 30% reservedata";
KØR;


                                       Ugentlig efterspørgsel og kandidatdrivere                                        

The GLMSELECT Procedure


Dependent Variable: UNITS Solgte enheder (ugentligt)


Number of Observations Used: 70               
Number of Observations Used for Validation: 30

Partition Fractions 

      Role  Fraction
----------  --------
  Training    0.7000
Validation    0.3000
   Testing    0.0000

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                        Forward Selection Summary                                                         

    Step    Action            Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)        PRESS
--------  --------  ---------


NOTE: PROC GLMSELECT data=demand

NOTE: PROC GLMSELECT statement used.


## 6. Fortolkning af resultaterne

Alle tre udvælgelsesstrategier genfinder **det samme sæt af sande efterspørgselsdrivere** og frasorterer alle lokkevariable. Læst direkte fra **Parameter Estimates**-tabellerne:

- **Pris** er den dominerende driver. Dens standardiserede koefficient i den trinvise model er **-0,70**, langt den største i talværdi, og den rå hældning ligger mellem **-27,8** (trinvis og LASSO) og **-28,6** (fremadrettet) enheder pr. dollar — næsten præcis de -28, dataene blev genereret med. **Konkurrentpris** flytter efterspørgslen den modsatte vej med omkring **+14,4** på tværs af alle tre modeller, den substitutionsadfærd en kategorichef forventer.
- **Reklameudgift** (ca. +0,062 enheder pr. dollar) og **e-mailvolumen** (ca. +1,2 enheder pr. udsendelse) løfter begge salget og kvantificerer medierespons.
- **Kampagner** og **helligdage** bærer de største hændelseseffekter: `Promo=No`-kontrasten ligger på omkring **-51 til -57** i forhold til en kampagneuge, og helligdagsløftet er omkring **+66 til +76** enheder. **Northeast-regionen** (ca. +49 til +55) og **Online-kanalen** (ca. +24 til +32) bærer positive grundlæggende effekter.
- Afgørende er det, at hver lokkevariabel — `temp_f`, `noise1`, `noise2`, `noise3` — **droppes** af trinvis og fremadrettet udvælgelse og udelukkes fra den CV-valgte LASSO-model. Hver metode genfandt det strukturelle signal og ignorerede støjen.

De tre metoder er ikke byte-for-byte identiske: trinvis (SBC) og den reservedata-validerede fremadrettede søgning lander på de samme otte effekter, mens den krydsvaliderede LASSO desuden beholder en lille `Region=Midwest`-kontrast (-8,3, standardfejl 8,3) — en forskel på støjniveau snarere end en substantiel uenighed. At en driverliste overlever trinvis SBC, krydsvalideret LASSO og reservedata-valideret fremadrettet udvælgelse er selve pointen: et fund, der er robust over for tre forskellige udvælgelsesfilosofier, er langt mere troværdigt end ét, der er tilpasset et enkelt kriterium. Med `OUTPUT OUT=demand_scored`, der indfanger prædikterede værdier og residualer, udvides samme arbejdsgang naturligt til at score næste kvartals planlagte pris- og kampagnekalender.